# 2. Managed パターン：Genie Agent とボリュームの content search

このノートブックは、画面の操作で RAG エージェントを作る **Managed パターン** のガイドである。コードはほとんど書かない。各 STEP の画像の赤枠と番号の順に、ワークスペースの画面を操作する。

```
UC マネージドボリューム（docs）── 添付 ──→ Genie Agent ──→ content search（自動でパース・索引作成・同期）
                                              │
                                   Agent mode で質問 → 引用付きの回答（ページ単位）
```

## このノートブックでやること
1. 前提（プレビューと権限）を確かめる
2. Genie Agent を作り、00 で作ったボリューム `docs` を添付する
3. Agent mode で質問し、回答の引用（ファイルとページ）を確かめる
4. 利用者と共有する

## 前提
- **00_データ準備** を実行し、マネージドボリュームに PDF がある
- ワークスペース管理者が **プレビュー** で「Analyze Files in Volumes with Genie Agents」を有効にしている（**Beta**、2026年10月時点）
- 利用者に **Workspace access** と **Databricks SQL access** の entitlement がある。Beta の間は、アカウントだけのユーザー（account-only user）は使えない
- 利用者に、添付するボリュームの `READ VOLUME` がある

## content search の主な制約
| 項目 | 内容 |
|---|---|
| 対応するボリューム | Unity Catalog の **マネージドボリュームのみ**（external volume と My Files は対象外） |
| 対応形式 | PDF、JPG/JPEG/PNG/TIFF/TIF、DOC/DOCX、PPT/PPTX、MD、TXT |
| ファイルの上限 | 100MB 超または 500 ページ超のファイルはスキップされる |
| 数の上限 | 1ボリューム1万ファイル、1 Agent 10ボリューム、1ワークスペース250ボリューム |
| モード | **Agent mode** だけがボリュームのファイルを使う。Chat mode は構造化データだけを扱う |
| 課金 | ファイルの準備（パース）は AI Functions、回答の推論は Genie の料金 |

> 画面の表示は 2026年10月時点の日本語 UI である。ラベルが変わっている場合は、最新のドキュメントで確かめる。

---

In [0]:
# 参考：00 で作ったボリュームの中身を確かめる（widget の値を 00 と合わせる）
dbutils.widgets.text("CATALOG_NAME", "main", "カタログ名")
import re
_user = spark.sql("SELECT current_user()").first()[0]
DEFAULT_SCHEMA = "rag_handson_" + re.sub(r"[^a-z0-9]+", "_", _user.split("@")[0].lower()).strip("_")
dbutils.widgets.text("SCHEMA_NAME", DEFAULT_SCHEMA, "スキーマ名（参加者ごと）")
dbutils.widgets.text("VOLUME_NAME", "docs", "ボリューム名")

VOLUME_PATH = f"/Volumes/{dbutils.widgets.get('CATALOG_NAME')}/{dbutils.widgets.get('SCHEMA_NAME')}/{dbutils.widgets.get('VOLUME_NAME')}"
display(dbutils.fs.ls(VOLUME_PATH))

## 2-1. 前提を確かめる

### STEP 1：プレビューを確かめる

**（ワークスペース管理者）**
①右上のユーザーアイコン →「プレビュー」を開き、絞り込み欄に「Analyze Files」と入力する。②「Analyze Files in Volumes with Genie Agents」が「オン」になっていることを確かめる。

![STEP 1](./images/managed-01.png)


→ オフの場合は、ワークスペース管理者がトグルをオンにする。反映まで数分かかる。

## 2-2. Genie Agent を作り、ボリュームを添付する

### STEP 2：新しく作る
①左サイドバーの「Genieエージェント」をクリックし、②右上の「新規」をクリックする。

![STEP 2](./images/managed-02.png)


→ Genie Agent の作成画面が表示される。

### STEP 3：ボリュームを選ぶ
①「データの接続」ダイアログで「ボリューム」をクリックし、②一覧から「docs　<カタログ>.<自分のスキーマ>」（例 `rag_handson_<ユーザー名>`）をクリックして、③「作成」をクリックする。

![STEP 3](./images/managed-03.png)


→ ボリュームを添付した Genie Agent ができ、content search が自動で有効になる。

> **注意**：content search はマネージドボリュームだけに対応する。

### STEP 4：ソースを確かめる
①右上の「設定する」をクリックして「ソース」タブを開き、②「docs」に「インデックスを作成中...」と表示されていることを確かめる。

![STEP 4](./images/managed-04.png)


→ content search がファイルのパースと索引作成を自動で進める。ファイル数によって数分〜数十分かかる。

### STEP 5：説明を書く
ソースの「docs」の鉛筆アイコンをクリックし、「説明」の鉛筆アイコンから「ボリュームの説明を編集」を開く。①何が入っていて、どんな質問に使うかを書き、②「保存」をクリックする。

![STEP 5](./images/managed-05.png)


→ Genie Agent が、この説明をもとにボリュームを使うかどうかを判断する。

## 2-3. Agent mode で質問し、引用を確かめる

### STEP 6：Agent modeで質問する
右のパネルで「docs」が「インデックス作成済み」になったことを確かめる。①入力欄の下で「エージェント」が選ばれていることを確かめ、②質問を入力して、③送信ボタンをクリックする。

![STEP 6](./images/managed-06.png)


→ Genie Agent が文書を検索しながら、多段階で推論して回答を作る。

> **注意**：「チャット」モードでは、ボリュームのファイルを使わない。

### STEP 7：回答と引用を見る
回答を読み、①本文の引用アイコン（フォルダーの形）で根拠の文書を確かめる。②回答の下の「1件のソース」をクリックする。

![STEP 7](./images/managed-07.png)


→ 回答に使った文書とページが表示される。

> **注意**：ページ単位の引用は Beta である。

### STEP 8：引用元を開く
表示された引用（例：「Automotive-SPICE-PAM_40_Japanese.pdf（p110）」）をクリックし、元の文書のページを確かめる。

![STEP 8](./images/managed-08.png)


→ 引用元のファイルとページが開き、回答の根拠を確かめられる。

## 2-4. 利用者と共有する

### STEP 9：利用者と共有する
右上の「共有」をクリックし、①利用者やグループを入力して権限を付ける。②「アクセス許可のある人」で現在の権限を確かめる。

![STEP 9](./images/managed-09.png)


→ 共有した利用者が Genie Agent を使える。データは利用者本人の権限で評価されるため、利用者にはボリュームの READ VOLUME も必要である。

## 2-5. 作った Genie Agent を外から使う

作った Genie Agent は、Genie の画面のほかに次の方法で使える。

| 使い方 | 内容 |
|---|---|
| **Genie One** | ビジネスユーザー向けの統合画面。Chat が、質問に合う Genie Agent を探して使う |
| **Agent mode API** | 外部のアプリや自作のエージェントから、Agent mode に質問を送り、推論の過程と引用付きの回答を受け取る（`POST /api/2.0/genie/agents/{agent_id}/responses`、GA） |
| **Genie Agent をツールとして呼ぶ** | Scratch パターンのエージェントに `agentbricks tools add genie-agent` で追加する（ハイブリッド構成） |

演習3の切り替えチャットアプリ（`apps/rag-handson-switcher`）は、Agent mode API を **質問したユーザー本人の権限** で呼ぶ。Agent の ID は、ブラウザの URL の `/genie/rooms/` の後ろの値である。

## まとめ

- ボリュームを添付するだけで、content search がパース・索引作成・同期を自動で行った
- Agent mode が文書を検索し、ページ単位の引用付きで回答した
- データへのアクセスは、常に質問したユーザー本人の Unity Catalog の権限で評価される

### Scratch パターンとの違い（振り返り）
| 観点 | Managed（このノートブック） | Scratch（1-1、1-2） |
|---|---|---|
| 作り方 | 画面の操作だけ | ノートブックのコードと CLI |
| パース・チャンク・埋め込み | 自動（中身は制御できない） | `ai_parse_document`、`ai_prep_search`、埋め込みモデルを選べる |
| 検索 | content search（自動同期） | AI Search（ANN／ハイブリッド、フィルタ、リランク） |
| 権限 | 質問者本人の UC 権限 | 既定はアプリのサービスプリンシパル。AI Search は行・列レベルの権限に非対応 |
| 品質管理 | Benchmarks、モニタリング | MLflow のトレースと評価 |

### 後片付け
Genie の一覧で、作った Genie Agent を削除する。ボリュームは README の「後片付け」の順にスキーマと一緒に消す。

### 参考リンク
- [Analyze files in volumes with a Genie Agent](https://docs.databricks.com/aws/ja/genie-agents/volumes)
- [Genie Agents の概念（Agent mode とデータアクセス）](https://docs.databricks.com/aws/ja/genie-agents/concepts)
- [Genie One](https://docs.databricks.com/aws/ja/genie-one/)
- [Genie Agents API](https://docs.databricks.com/aws/ja/genie-agents/conversation-api)